# Problem Statement :   CardioGood Fitness Data Analysis

## Context
 The market research team at AdRight is assigned the task to identify the profile of the typical customer for each treadmill product offered by CardioGood Fitness. The market research team decides to investigate whether there are differences across the product lines with respect to customer characteristics. The team decides to collect data on individuals who purchased a treadmill at a CardioGood Fitness retail store at any time in the past three months. The data is stored in the CardioGoodFitness.csv file.

## Objective
 Perform descriptive analysis to create a customer profile for each CardioGood Fitness treadmill product line.

## Data Dictionary

**The team identified the following customer variables to study:**

* Product: Product purchased - TM195, TM498, or TM798
* Gender: Male or Female
* Age: Age of the customer in years
* Education: Education of the customer in years
* MaritalStatus: Single or partnered
* Income: Annual household income
* Usage: The average number of times the customer plans to use the treadmill each week;
* Miles: The average number of miles the customer expects to walk/run each week;
* Fitness: Self-rated fitness on a 1-to-5 scale, where 1 is the poor shape and 5 is excellent shape.

## Questions to explore
* What are the different types of variables in the data?
* What is the distribution of different variables in the data?
* Which product is more popular among males or females?
* Is the product purchase affected by the marital status of the customer?
* Is there a significant correlation among some of the variables?
* What is the distribution of the average number of miles for each product?

## 0. Setup

In [ ]:
#load necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# load data set (CSV sits in the same folder as this notebook)
mydata = pd.read_csv("CardioGoodFitness.csv")

## 1. Data overview & univariate analysis

In [ ]:
# read data
mydata.head()

In [ ]:
#generate Analytical summary of all the data
mydata.describe(include = 'all')

In [ ]:
print(f"Number of rows and columns: {mydata.shape}")

# Detailed structural summary of the data
mydata.info()

In [ ]:
mydata.hist(figsize = (20,30))

In [ ]:
sns.boxplot(x ='Gender', y ='Age',data = mydata )
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(x='Product',y ='Age', data = mydata)
plt.tight_layout()
plt.show()

## 2. Bivariate analysis: product vs. gender and marital status

In [ ]:
pd.crosstab(mydata['Product'],mydata['Gender'])

In [ ]:
pd.crosstab(mydata['Product'],mydata['MaritalStatus'])

In [ ]:
sns.countplot(x='Product',hue='Gender',data = mydata)
plt.tight_layout()
plt.show()

In [ ]:
pd.pivot_table(mydata, index=['Product','Gender'],
               columns=['MaritalStatus'],aggfunc='count')

In [ ]:
pd.pivot_table(mydata,'Income',index=['Product' ,'Gender'],
               columns = ['MaritalStatus'])

In [ ]:
pd.pivot_table(mydata,'Miles',index=['Product','Gender'],columns = ['MaritalStatus'])

## 3. Multivariate view & distributions

In [ ]:
sns.pairplot(mydata);

In [ ]:
mydata['Age'].std()

In [ ]:
mydata['Age'].mean()

In [ ]:
sns.histplot(mydata['Age'], kde=True)
plt.show()

In [ ]:
mydata.hist(by = 'Gender',column = 'Income')

## 4. Correlation analysis

In [ ]:
# Compute correlation matrix only for numeric columns
corr = mydata.corr(numeric_only=True)
corr

In [ ]:
cov = mydata.cov(numeric_only=True)
cov

In [ ]:
sns.heatmap(corr,annot=True);

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Product popularity by Gender
sns.countplot(data=mydata, x='Product', hue='Gender', ax=axes[0], palette='Set2')
axes[0].set_title('Product Popularity by Gender')
axes[0].set_xlabel('Product Model')
axes[0].set_ylabel('Count')

# Plot 2: Product popularity by Marital Status
sns.countplot(data=mydata, x='Product', hue='MaritalStatus', ax=axes[1], palette='Pastel1')
axes[1].set_title('Product Popularity by Marital Status')
axes[1].set_xlabel('Product Model')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Regression: predicting weekly miles

In [ ]:
#simple Linear Regression

#Load Function from sklearn
from sklearn import linear_model

#creat Linear regression object
regr = linear_model.LinearRegression()
y =mydata['Miles']
x = mydata[['Usage', 'Fitness']]

#fit the model
regr.fit(x,y)

print('R-squared:', round(regr.score(x, y), 3))

In [ ]:
print('Coefficients (Usage, Fitness):', regr.coef_.round(2))

In [ ]:
print('Intercept:', round(regr.intercept_, 2))

In [ ]:
# Example: customer planning 4 uses/week with fitness rating 4
print('Predicted weekly miles:', round(regr.predict(pd.DataFrame({'Usage':[4],'Fitness':[4]}))[0], 1))

## Conclusions & Customer Profiles

**Regression:** `Miles ≈ -56.74 + 20.21 × Usage + 27.21 × Fitness` (R² ≈ 0.72). Planned usage and self-rated fitness together explain about 72% of the variation in weekly miles.

| Product | Customers | Avg age | Avg education (yrs) | Avg income | Avg usage/wk | Avg fitness | Avg miles/wk | Gender split (F/M) |
|---|---|---|---|---|---|---|---|---|
| TM195 | 80 | 28.6 | 15.0 | $46,418 | 3.1 | 3.0 | 82.8 | 50% / 50% |
| TM498 | 60 | 28.9 | 15.1 | $48,974 | 3.1 | 2.9 | 87.9 | 48% / 52% |
| TM798 | 40 | 29.1 | 17.3 | $75,442 | 4.8 | 4.6 | 166.9 | 18% / 82% |

**Answers to the questions explored**
* **Gender:** TM195 and TM498 are bought about equally by men and women; TM798 is bought mostly by men (~82%).
* **Marital status:** About 60% of buyers of every product are partnered, so marital status does not noticeably change which product is bought.
* **Correlations:** Usage, Fitness and Miles are strongly related to one another; Income is positively related to Education and to the more advanced product.
* **Miles by product:** TM798 customers run roughly twice as many miles per week (median 160) as TM195/TM498 customers (median 85).

**Customer profiles**
* **TM195 (entry level):** ~29 years old, mid income (~$46K), average fitness, moderate use. Men and women equally.
* **TM498 (mid level):** nearly identical to TM195 (slightly higher income); similar usage and fitness.
* **TM798 (premium):** mostly male, highly educated, high income (~$75K), very fit, uses the treadmill ~5 times a week.

**Recommendation:** Position TM798 toward fit, high-income, frequent users (often male), and target TM195/TM498 at broader, more casual audiences, including women.

*Caveat:* The dataset has only 180 customers from a three-month window, so these findings are descriptive, not causal.